In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle



In [ ]:
data = pd.read_csv('Churn_Modelling.csv')

data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo= OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography',axis=1), geo_encoder_df],axis=1)


x = data.drop('Exited',axis=1)
y = data['Exited']

x_train,x_test,y_train,y_test= train_test_split(x,y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train= scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open("onehot_encoder_geo.pkl",'wb') as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)
    

In [30]:
# define a function to create a model and try different params(keras classifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(x_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])

    return model



In [31]:
#  create a keras classifier

model = KerasClassifier(layers=1, neurons=32,build_fn=create_model, epochs=50, batch_size=10, verbose=0)

In [24]:
# define the grid search params
param_grid = {
    'neurons' :[16, 32, 64, 128],
    'layers': [1,2],
    'epochs': [50,100]
}

In [33]:
#  perform grip search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(x_train, y_train)


# print the best paramtetars
print("best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

KeyboardInterrupt: 